# 🎨 face2sketch — Phase 2: Finetune → Caricature (Kaggle)

**Load Phase 1 G weights, finetune on TwitterPicasso (184 pairs) for caricature style.**

### Prerequisites
- **GPU:** T4 x2 — enable in Notebook → Accelerator → GPU
- **Internet:** Enable in Notebook → Internet → On
- **Dataset:** Add `face2sketch` dataset as input (contains data.zip + pix2pix_best.pt)

### Strategy (v2)
- **λ_L1=200** — L1 does the heavy lifting (D dies early on 184 pairs)
- **50-epoch L1-only warmup** — build caricature structure first, then add adversarial
- **LR=1e-4** — higher to escape Phase 1 weights faster
- **Batch=32** — T4 x2 has 32GB VRAM (2× vs Colab's single T4)

In [ ]:
# @title 1. Clone Repo + Install
import os, sys
BASE = '/kaggle/working'
os.chdir(BASE)

!git clone https://github.com/weseegod/face2sketch.git {BASE}/face2sketch 2>/dev/null || true
os.chdir(f'{BASE}/face2sketch')
!git pull origin main

!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples outputs

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"Working dir: {os.getcwd()}")
print('\n✅  Setup complete.')

In [ ]:
# @title 2. Extract Data + Checkpoint from Kaggle Dataset
import glob, os, shutil

# Search for files anywhere under /kaggle/input/
INPUT = '/kaggle/input'

data_zip = None
ckpt_path = None
data_already_extracted = False

for root, dirs, files in os.walk(INPUT):
    for f in files:
        path = os.path.join(root, f)
        if f == 'data.zip' and not data_zip:
            data_zip = path
        if f == 'pix2pix_best.pt' and not ckpt_path:
            ckpt_path = path
    # Check if data/ is already extracted in the dataset
    if 'finetune' in dirs and 'dataset' in dirs:
        data_already_extracted = True
        extracted_data_root = root

DEST = '/kaggle/working/face2sketch'

# --- Extract or copy data ---
if data_zip:
    print(f'📦  Extracting {data_zip}...')
    get_ipython().system('unzip -o {data_zip} -d {DEST}/ 2>&1 | tail -3')
elif data_already_extracted:
    print(f'📂  Data already extracted — copying from {extracted_data_root}')
    for sub in ['dataset', 'finetune', 'test']:
        src = os.path.join(extracted_data_root, 'data', sub)
        dst = os.path.join(DEST, 'data', sub)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print(f'    data/{sub}/ ✅')
else:
    print('❌  No data found.')

# --- Copy checkpoint ---
os.makedirs(f'{DEST}/checkpoints', exist_ok=True)
if ckpt_path:
    shutil.copy(ckpt_path, f'{DEST}/checkpoints/pix2pix_best.pt')
    print(f'📂  Checkpoint: pix2pix_best.pt ✅')
else:
    print('❌  pix2pix_best.pt not found')

# --- Verify ---
ft_dir = f'{DEST}/data/finetune'
ft_photos = len(os.listdir(f'{ft_dir}/photos')) if os.path.isdir(f'{ft_dir}/photos') else 0
ft_sketch = len(os.listdir(f'{ft_dir}/sketches')) if os.path.isdir(f'{ft_dir}/sketches') else 0
ckpt_ok = os.path.exists(f'{DEST}/checkpoints/pix2pix_best.pt')

print(f'\n✅  Finetune pairs: {min(ft_photos, ft_sketch)}')
print(f'✅  Phase 1 ckpt:  {"YES" if ckpt_ok else "NO"}')


In [ ]:
# @title 3. Finetune — 100 epochs (~30 min T4 x2)
import os
assert torch.cuda.is_available(), "⚠️  Enable GPU in Notebook settings"

BASE = '/kaggle/working/face2sketch'
os.chdir(BASE)

assert os.path.exists('checkpoints/pix2pix_best.pt'), "❌ Missing Phase1 checkpoint!"
assert os.path.isdir('data/finetune'), "❌ Missing finetune data!"

print("🎯  Finetuning Phase 1 G on TwitterPicasso caricatures")
print("    λ_L1=200 | 50-epoch L1 warmup | LR=1e-4 | Batch=32")
print(f"    Dataset: 184 pairs  |  Epochs: 100\n")

!python src/train.py --mode finetune \
    --config configs/pix2pix_phase2.yaml \
    --finetune checkpoints/pix2pix_best.pt \
    --device cuda \
    --batch-size 32 \
    --name phase2_

print("\n✅  Finetuning complete → checkpoints/phase2_best.pt")

In [ ]:
# @title 4. Evaluate — Phase 2 Gate Check
import os, glob

BASE = '/kaggle/working/face2sketch'
os.chdir(BASE)

ckpt = 'checkpoints/phase2_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/phase2_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No Phase 2 checkpoint found.')
elif not os.path.exists('data/test/photos'):
    print('⚠️  No test data found.')
else:
    !python src/evaluate.py --checkpoint {ckpt} --mode quick --device cuda

    print('\n📊  Comparing Phase 1 vs Phase 2 outputs...')
    p1_ckpt = 'checkpoints/pix2pix_best.pt'
    if os.path.exists(p1_ckpt):
        !python src/evaluate.py --checkpoint {p1_ckpt} \
            --checkpoint2 {ckpt} --mode compare --device cuda
    print('   ✅  Comparison: outputs/phase1_vs_phase2.png')

In [ ]:
# @title 5. Save Results to Kaggle Output
import os, shutil

BASE = '/kaggle/working/face2sketch'
OUT = '/kaggle/working'

for item in ['checkpoints', 'samples', 'outputs']:
    src = os.path.join(BASE, item)
    dst = os.path.join(OUT, item)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✅  Copied {item}/ → /kaggle/working/{item}/")

print("\n--- Kaggle Output (/kaggle/working/) ---")
for root, dirs, files in os.walk(OUT):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        rel = os.path.relpath(path, OUT)
        label = f"{size/(1024*1024):.0f}MB" if size > 1024*1024 else f"{size/1024:.0f}KB"
        print(f"  {rel}  ({label})")

print("\n🎉  All done! Download from Kaggle Output tab.")